In [2]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('../dataset/autoMpg.csv')

col = df['mpg'].dropna()
n   = len(col)
mu  = col.mean()
se  = stats.sem(col)          # errore standard = std / sqrt(n)

# IC 95% con distribuzione NORMALE (z) — valida asintoticamente per n grande
z = stats.norm.ppf(0.975)
ci_z_low  = mu - z * se
ci_z_high = mu + z * se

# IC 95% con distribuzione t di STUDENT — corretta per qualsiasi n
ci_t_low, ci_t_high = stats.t.interval(0.95, df=n-1, loc=mu, scale=se)

print(f"n                    = {n}")
print(f"Media                = {mu:.4f}")
print(f"Errore standard (SE) = {se:.4f}")
print()
print(f"IC 95% Normale (z):  [{ci_z_low:.4f},  {ci_z_high:.4f}]  (z = {z:.4f})")
print(f"IC 95% Student (t):  [{ci_t_low:.4f},  {ci_t_high:.4f}]  (t = {stats.t.ppf(0.975, df=n-1):.4f})")


n                    = 398
Media                = 24.1251
Errore standard (SE) = 0.7328

IC 95% Normale (z):  [22.6889,  25.5614]  (z = 1.9600)
IC 95% Student (t):  [22.6845,  25.5658]  (t = 1.9660)


In [7]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(42)

# Gruppo A: studenti senza tutoraggio (media 70)
# Gruppo B: studenti con tutoraggio    (media 76)
gruppo_A = rng.normal(loc=70, scale=10, size=40)
gruppo_B = rng.normal(loc=76, scale=10, size=40)

print(f"Media A = {gruppo_A.mean():.2f}")
print(f"Media B = {gruppo_B.mean():.2f}")
print()

# H0: mu_B <= mu_A  (il tutoraggio non migliora i voti)
# H1: mu_B >  mu_A  (il tutoraggio migliora i voti) → test unilaterale destro
t_stat, p_two = stats.ttest_ind(gruppo_A, gruppo_B)

# p-value unilaterale: se t < 0 (B > A) → p_one = p_two / 2
p_one = p_two / 2 if t_stat < 0 else 1 - p_two / 2

print(f"Statistica t         = {t_stat:.4f}  (negativo → B > A)")
print(f"p-value bilaterale   = {p_two:.4f}  (H1: A ≠ B)")
print(f"p-value unilaterale  = {p_one:.4f}  (H1: B > A)")
print()

alpha = 0.05
if p_one < alpha:
    print(f"Rifiutiamo H0 (p={p_one:.4f} < α={alpha}): B è significativamente più alto di A")
else:
    print(f"Non rifiutiamo H0 (p={p_one:.4f} >= α={alpha}): differenza non significativa")


Media A = 70.38
Media B = 76.13

Statistica t         = -3.2995  (negativo → B > A)
p-value bilaterale   = 0.0015  (H1: A ≠ B)
p-value unilaterale  = 0.0007  (H1: B > A)

Rifiutiamo H0 (p=0.0007 < α=0.05): B è significativamente più alto di A


In [8]:
import pandas as pd
from scipy import stats

# Tabella di contingenza: preferenza di prodotto per fascia d'età
# Righe = fasce d'età, Colonne = prodotto scelto
osservati = pd.DataFrame(
    [[30, 10, 15],
     [25, 30, 20],
     [10, 25, 35]],
    index   = ['18-30', '31-50', '51+'],
    columns = ['Prodotto A', 'Prodotto B', 'Prodotto C']
)

print("Frequenze osservate:")
print(osservati)
print()

# H0: fascia d'età e preferenza di prodotto sono INDIPENDENTI
# H1: esiste un'associazione tra le due variabili
chi2, p, dof, attesi = stats.chi2_contingency(osservati)

print("Frequenze attese (sotto H0):")
print(pd.DataFrame(attesi, index=osservati.index, columns=osservati.columns).round(2))
print()

print(f"Chi² = {chi2:.4f}")
print(f"Gradi di libertà = {dof}  [(righe-1)×(colonne-1) = {osservati.shape[0]-1}×{osservati.shape[1]-1}]")
print(f"p-value = {p:.4f}")
print()

alpha = 0.05
if p < alpha:
    print(f"Rifiutiamo H0 (p={p:.4f} < α={alpha}): la preferenza dipende dalla fascia d'età")
else:
    print(f"Non rifiutiamo H0 (p={p:.4f} >= α={alpha}): nessuna associazione significativa")


Frequenze osservate:
       Prodotto A  Prodotto B  Prodotto C
18-30          30          10          15
31-50          25          30          20
51+            10          25          35

Frequenze attese (sotto H0):
       Prodotto A  Prodotto B  Prodotto C
18-30       17.88       17.88       19.25
31-50       24.38       24.38       26.25
51+         22.75       22.75       24.50

Chi² = 27.3027
Gradi di libertà = 4  [(righe-1)×(colonne-1) = 2×2]
p-value = 0.0000

Rifiutiamo H0 (p=0.0000 < α=0.05): la preferenza dipende dalla fascia d'età
